# Implement Grouped Query Attention from Scratch
### 🧠 Problem Statement
Standard Multi-Head Attention (MHA) gives every attention head its own query, key and value projection. During autoregressive decoding the keys and values of every past token are cached, so the **KV cache** (and the memory bandwidth spent reading it at every step) grows with the number of K/V heads.

**Grouped Query Attention (GQA)** (Ainslie et al., 2023; used by Llama 2 70B, Llama 3, Mistral) keeps `num_query_heads` query heads but only `num_query_groups` key/value heads. Each K/V head is shared by a contiguous group of `g = num_query_heads / num_query_groups` query heads: query head `h` uses K/V head `h // g`. This shrinks the KV cache by a factor of `g`; the attention FLOPs stay the same. `num_query_groups == num_query_heads` is ordinary MHA, and `num_query_groups == 1` is multi-query attention (MQA).

Your task is to **implement the GQA forward pass from scratch** and validate it against PyTorch.

---

### ✅ Requirements

1. **Signature**: `grouped_query_attention(q, k, v, num_query_heads, num_query_groups, W_q, W_k, W_v, W_o, mask=None)`
   - `q`, `k`, `v`: `(batch_size, seq_len, d_model)`; `d_head = d_model // num_query_heads`.
   - `W_q`, `W_o`: `(d_model, d_model)`; `W_k`, `W_v`: `(num_query_groups * d_head, d_model)`. All in `nn.Linear` layout: a projection is `x @ W.T`.
   - `num_query_groups` must divide `num_query_heads`; raise an error otherwise.

2. **Share the K/V heads**
   - Split Q into `num_query_heads` heads and K/V into `num_query_groups` heads.
   - Expand K/V to `num_query_heads` heads with `repeat_interleave(g, dim=1)` so query head `h` meets K/V head `h // g`. (`.repeat` would tile them instead and pair head `h` with K/V head `h % num_query_groups`.)

3. **Compute Attention**
   - `Q @ Kᵀ / sqrt(d_head)`, optional boolean `mask` broadcastable to `(batch, heads, seq, seq)` with **`True` = may attend**, softmax, weight `V`.
   - Concatenate the heads and apply `W_o`. Output shape `(batch_size, seq_len, d_model)`.

4. **Validate**
   - `num_query_groups == num_query_heads`: compare with `nn.MultiheadAttention(bias=False, batch_first=True)` using its weights (`in_proj_weight.chunk(3)`, `out_proj.weight`).
   - `num_query_groups < num_query_heads`: compare with `F.scaled_dot_product_attention(..., enable_gqa=True)` applied to your own projections.

---

### 📏 Constraints

- ✅ Use only PyTorch (no xformers or Hugging Face); do not call `F.scaled_dot_product_attention` inside your function.
- ✅ Output shape must be `(batch_size, seq_len, d_model)`.
- ✅ Support optional attention masking.

---

<details>
  <summary>💡 Hint</summary>

  - `W_k` and `W_v` produce only `num_query_groups * d_head` features. View them as `(batch, seq, num_query_groups, d_head)` and transpose to `(batch, num_query_groups, seq, d_head)`.
  - `K.repeat_interleave(num_query_heads // num_query_groups, dim=1)` gives every query head its K/V head.
  - From there it is ordinary multi-head attention: scores, mask, softmax, `@ V`, merge heads, `W_o`.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_query_heads = 4
d_head = d_model // num_query_heads

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

In [ ]:
def grouped_query_attention(q, k, v, num_query_heads, num_query_groups,
                            W_q, W_k, W_v, W_o, mask=None):
    """
    Implements Grouped Query Attention (GQA).

    Queries keep `num_query_heads` heads, but keys and values only have
    `num_query_groups` heads; K/V head j is shared by the
    g = num_query_heads // num_query_groups query heads j*g .. j*g + g - 1.
    With num_query_groups == num_query_heads this is ordinary multi-head
    attention, and with num_query_groups == 1 it is multi-query attention.

    Args:
        q, k, v (Tensor): (batch_size, seq_len, d_model)
        num_query_heads (int): number of query heads; d_head = d_model // num_query_heads
        num_query_groups (int): number of key/value heads; must divide num_query_heads
        W_q, W_o (Tensor): (d_model, d_model)
        W_k, W_v (Tensor): (num_query_groups * d_head, d_model)
            All weights are in nn.Linear layout: a projection is x @ W.T
        mask (Tensor, optional): bool, broadcastable to (batch, heads, seq, seq);
            True = may attend

    Returns:
        Tensor: (batch_size, seq_len, d_model)
    """
    batch_size, seq_len, d_model = q.shape
    if d_model % num_query_heads or num_query_heads % num_query_groups:
        raise ValueError("num_query_heads must divide d_model, and num_query_groups "
                         "must divide num_query_heads")
    d_head = d_model // num_query_heads
    group_size = num_query_heads // num_query_groups

    # Queries get every head; keys/values only one head per group.
    Q = F.linear(q, W_q).view(batch_size, seq_len, num_query_heads, d_head).transpose(1, 2)
    K = F.linear(k, W_k).view(batch_size, seq_len, num_query_groups, d_head).transpose(1, 2)
    V = F.linear(v, W_v).view(batch_size, seq_len, num_query_groups, d_head).transpose(1, 2)

    # K/V head j serves query heads j*g .. j*g + g - 1: (batch, num_query_heads, seq, d_head)
    K = K.repeat_interleave(group_size, dim=1)
    V = V.repeat_interleave(group_size, dim=1)

    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_head)
    if mask is not None:
        scores = scores.masked_fill(~mask.bool(), float("-inf"))
    attn_weights = F.softmax(scores, dim=-1)
    output = attn_weights @ V

    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    return F.linear(output, W_o)

In [ ]:
# 1. With as many K/V groups as query heads, GQA is MHA: compare with nn.MultiheadAttention.
mha = nn.MultiheadAttention(d_model, num_query_heads, bias=False, batch_first=True)
W_q, W_k, W_v = mha.in_proj_weight.chunk(3)
W_o = mha.out_proj.weight
with torch.no_grad():
    out = grouped_query_attention(q, k, v, num_query_heads, num_query_heads, W_q, W_k, W_v, W_o)
    ref, _ = mha(q, k, v, need_weights=False)
assert torch.allclose(out, ref, atol=1e-6)
print("num_query_groups=4 (MHA): matches nn.MultiheadAttention")

# 2. Fewer K/V heads: compare with SDPA's enable_gqa, which also maps query head h
#    to K/V head h // (num_query_heads // num_query_groups).
def split(x, heads):
    return x.view(batch_size, seq_len, heads, d_head).transpose(1, 2)

causal = torch.ones(seq_len, seq_len, dtype=torch.bool).tril()
for num_query_groups in (1, 2):
    W_k = torch.randn(num_query_groups * d_head, d_model) / d_model ** 0.5
    W_v = torch.randn(num_query_groups * d_head, d_model) / d_model ** 0.5
    with torch.no_grad():
        out = grouped_query_attention(q, k, v, num_query_heads, num_query_groups,
                                      W_q, W_k, W_v, W_o, mask=causal)
        ref = F.scaled_dot_product_attention(
            split(q @ W_q.T, num_query_heads), split(k @ W_k.T, num_query_groups),
            split(v @ W_v.T, num_query_groups), is_causal=True, enable_gqa=True)
        ref = ref.transpose(1, 2).reshape(batch_size, seq_len, d_model) @ W_o.T
    assert torch.allclose(out, ref, atol=1e-6)
    print(f"num_query_groups={num_query_groups}: matches SDPA(enable_gqa=True); "
          f"W_k is {tuple(W_k.shape)} instead of {(d_model, d_model)}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("grouped-query-attention", grouped_query_attention=grouped_query_attention)